# FinSight | Financial headline sentiment
**From raw data to a defensible baseline — a reproducible, executable research notebook**

| | |
|---|---|
| **Objective** | Classify financial headlines as negative, neutral or positive. |
| **Dataset** | [Financial PhraseBank (Hugging Face)](https://huggingface.co/datasets/lmassaron/FinancialPhraseBank). |
| **Approach** | Conflict-aware cleaning → deduplicated splits → TF–IDF + logistic regression → validation/test evaluation → examples. |
| **Scope** | Research demonstration only; **not a financial-return predictor or trading system**. |

**Run this from the repository's virtual environment** (Python 3.10+):

```bash
cd finsight-ml-platform
git switch portfolio-review-2026-11-20
python -m pip install -e '.[data,notebook]'
python -m jupyter lab notebooks/FinSight_Financial_Sentiment.ipynb
```

Select your `.venv` kernel, then choose **Run → Run All Cells**. An internet connection is required to download the dataset on the first run. If you only want to test the notebook's mechanics offline, launch Jupyter with `FINSIGHT_NOTEBOOK_SOURCE=synthetic` (its results **are not real research findings**).

> **Integrity controls:** All examples for the same normalised headline are excluded if source labels conflict. Identical headlines are prevented from crossing train/validation/test splits. Test data is not used to choose model settings. Saved numerical results are generated at run time, never hard-coded.

## 1 · Set up the project environment
Find the repository root regardless of whether Jupyter starts in the root or `notebooks/`. Import modelling functions from FinSight's existing Python package rather than maintaining divergent notebook-only logic.

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = next(
    (candidate for location in [Path.cwd(), *Path.cwd().parents]
     for candidate in [location, location.parent]
     if (candidate / "pyproject.toml").exists() and (candidate / "src" / "finsight").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the FinSight repository (or its notebooks folder).")
sys.path.insert(0, str(ROOT / "src"))

from finsight.data.splits import LABEL_NAMES, load_splits, prepare_splits
from finsight.inference import predict_text
from finsight.models.reproducible import fit_and_evaluate

MODE = os.getenv("FINSIGHT_NOTEBOOK_SOURCE", "huggingface").strip().lower()
if MODE not in {"huggingface", "synthetic"}:
    raise ValueError("FINSIGHT_NOTEBOOK_SOURCE must be 'huggingface' or 'synthetic'")
print(f"Repository: {ROOT}")
print(f"Dataset mode: {MODE}")
print(f"Python: {sys.version.split()[0]}")

## 2 · Load data and audit the labels
The data provider offers train, validation, and test splits. We preserve those when all three are present. Financial PhraseBank can contain the **same normalised headline with different labels**; in that case, all copies of that headline are excluded and the counts are recorded. This is an explicit choice that may affect measured performance.

For `synthetic` mode, the notebook generates deliberately simple artificial examples (including one conflicting annotation) so you can verify the workflow without downloading data. **Never report synthetic scores as model performance.**

In [ ]:
def make_synthetic_data():
    # Offline functionality check only; these phrases are intentionally easy.
    phrases = {
        0: "reports declining profit and higher losses",
        1: "announces a routine shareholder meeting",
        2: "reports rising profit and record growth",
    }
    data = {}
    for split in ("train", "validation", "test"):
        data[split] = pd.DataFrame([
            {
                "sentence": f"{split} business {i} {phrases[i % 3]} for the period",
                "label": i % 3,
            }
            for i in range(36)
        ])
    conflict = data["train"].iloc[0].to_dict()
    conflict["label"] = 1  # Same sentence with incompatible annotation
    data["train"] = pd.concat([data["train"], pd.DataFrame([conflict])], ignore_index=True)
    return data

if MODE == "huggingface":
    splits, metadata = load_splits()
else:
    splits, metadata = prepare_splits(make_synthetic_data())
    metadata["dataset"] = "synthetic_offline_fixture_NOT_REAL_DATA"

source_counts = metadata["original_source_rows_by_split"]
final_counts = metadata["rows_after_cross_split_dedup"]
audit_table = pd.DataFrame({
    "source_rows": pd.Series(source_counts),
    "conflicting_label_rows_removed": pd.Series(metadata["ambiguous_source_rows_excluded_by_split"]),
    "final_rows": pd.Series(final_counts),
})
print("MODE:", MODE.upper(), "—", "SYNTHETIC SCORES ARE NOT RESEARCH RESULTS" if MODE == "synthetic" else "REAL DATASET")
print("Unique conflicting headlines excluded:", metadata["ambiguous_unique_headlines_excluded"])
display(audit_table)
assert all(not frame.empty for frame in splits.values())

## 3 · Inspect the cleaned class balance
Class imbalance matters: plain accuracy can look strong even when the model mostly predicts the common class. We will report **macro-F1**, **balanced accuracy**, and a majority-class comparator.

In [ ]:
class_balance = pd.DataFrame({
    split: frame["label"].map(LABEL_NAMES).value_counts().reindex(LABEL_NAMES.values(), fill_value=0)
    for split, frame in splits.items()
})
class_balance.index.name = "sentiment"
display(class_balance)
ax = class_balance.T.plot(kind="bar", figsize=(9, 4), rot=0)
ax.set(title="Sentiment labels after cleaning", xlabel="Dataset split", ylabel="Number of headlines")
ax.legend(title="Label")
plt.tight_layout()
plt.show()

## 4 · Train the reference model and compare baselines
The existing `fit_and_evaluate` function fits the **TF–IDF unigram/bigram + class-balanced logistic regression** pipeline only on the training split, and reports validation/test scores alongside a dummy classifier that always predicts the most frequent training label.

**Interpretation rule:** A positive sentiment label concerns the *wording* of a headline, **not** the future stock-price direction.

In [ ]:
model, scores = fit_and_evaluate(splits)
rows = []
for model_name in ("model", "majority_baseline"):
    for split_name in ("validation", "test"):
        metrics = scores[model_name][split_name]
        rows.append({
            "model": "TF-IDF + logistic regression" if model_name == "model" else "Majority class baseline",
            "split": split_name,
            "n": metrics["n_examples"],
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "balanced_accuracy": metrics.get("balanced_accuracy", np.nan),
        })
performance = pd.DataFrame(rows)
display(performance.style.format({"accuracy": "{:.3f}", "macro_f1": "{:.3f}", "balanced_accuracy": "{:.3f}"}))
print("Note: this notebook evaluates a predefined baseline; no test-set tuning is performed.")

## 5 · Confusion matrices and diagnostic errors
A confusion matrix shows *which* sentiment classes are confused, not just an overall number. Examine errors cautiously: financial headlines may legitimately admit more than one interpretation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, split_name in zip(axes, ("validation", "test")):
    cm = np.asarray(scores["model"][split_name]["confusion_matrix"])
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(LABEL_NAMES.values())).plot(
        ax=ax, colorbar=False, cmap="Blues", values_format="d"
    )
    ax.set_title(f"{split_name.title()} — TF-IDF + logistic regression")
plt.tight_layout()
plt.show()

test = splits["test"].copy()
test["predicted_label"] = model.predict(test["sentence"])
test["confidence"] = model.predict_proba(test["sentence"]).max(axis=1)
test["actual"] = test["label"].map(LABEL_NAMES)
test["predicted"] = test["predicted_label"].map(LABEL_NAMES)
errors = test.loc[test["label"] != test["predicted_label"], ["sentence", "actual", "predicted", "confidence"]]
print(f"Misclassifications: {len(errors)} of {len(test)} test examples")
# Display results locally; notebook saved on GitHub has no committed outputs.
display(errors.sort_values("confidence", ascending=False).head(10).reset_index(drop=True))

## 6 · Explore which terms influence predictions
Coefficients are **model features**, not causal drivers of returns. High positive coefficients for a sentiment class indicate that the model associates those words with that label in the training corpus.

In [ ]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]
features = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, label_id in zip(axes, classifier.classes_):
    row = list(classifier.classes_).index(label_id)
    coefs = classifier.coef_[row]
    top = np.argsort(coefs)[-10:]
    ax.barh(features[top], coefs[top])
    ax.set_title(f"Top features: {LABEL_NAMES[int(label_id)]}")
    ax.set_xlabel("Logistic-regression coefficient")
plt.tight_layout()
plt.show()

## 7 · Try your own headline
Change `MY_HEADLINE` and re-run this cell. The numbers are conditional **model outputs**, not calibrated investment probabilities. Fictional headlines are suitable for feature testing, but should not be mistaken for real corporate events.

In [ ]:
MY_HEADLINE = (
    "Tesla shares surge after the company raises its annual profit forecast"
)
result = predict_text(model, MY_HEADLINE)
print("Headline:", MY_HEADLINE)
print("Prediction:", result["label"])
display(pd.DataFrame([result["probabilities"]], index=["model score"]).T.rename(columns={"model score": "score"}))

## 8 · Prototype a company-headline research sheet
This is a **manual input worksheet**, not live data collection, securities valuation, or a full company report. Replace the fictional examples with actual headlines and independently verify each title, date and URL. FinSight only scores the supplied text; it cannot infer the issuer's financial health from a headline alone.

In [ ]:
company_headlines = pd.DataFrame([
    {
        "company": "ExampleCo",
        "headline": "ExampleCo reports strong revenue growth and a higher profit outlook",
        "source_url": "FICTIONAL EXAMPLE — REPLACE",
    },
    {
        "company": "ExampleCo",
        "headline": "ExampleCo warns of weaker sales and increased operating costs",
        "source_url": "FICTIONAL EXAMPLE — REPLACE",
    },
    {
        "company": "ExampleCo",
        "headline": "ExampleCo announces its annual shareholder meeting date",
        "source_url": "FICTIONAL EXAMPLE — REPLACE",
    },
])
company_headlines["sentiment"] = [predict_text(model, headline)["label"] for headline in company_headlines["headline"]]
company_headlines["positive_score"] = [predict_text(model, headline)["probabilities"]["positive"] for headline in company_headlines["headline"]]
display(company_headlines)
print("Reminder: average sentiment scores do not provide a valid stock-return forecast.")

## 9 · Save your trained model and evaluation report
A **real-dataset run** stores the model and audit metrics in the repository's ignored `artifacts/` directory, enabling CLI usage (`python -m finsight predict ...`) without retraining. Synthetic-mode runs intentionally **do not** write model artifacts, preventing them from being confused with genuine results. The model is a pickle-based `joblib` artifact: never load one from an untrusted party.

In [ ]:
if MODE == "huggingface":
    output = ROOT / "artifacts"
    output.mkdir(exist_ok=True)
    report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source": metadata,
        "evaluation": scores,
        "model_description": "TF-IDF word 1-2grams, class-balanced logistic regression",
        "limitations": "Financial text classification only; no investment return prediction.",
        "versions": {
            "python": sys.version.split()[0],
            "scikit_learn": version("scikit-learn"),
            "pandas": version("pandas"),
        },
    }
    (output / "evaluation.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    dump({"pipeline": model, "label_names": LABEL_NAMES}, output / "sentiment_model.joblib")
    print("Saved model:", output / "sentiment_model.joblib")
    print("Saved report:", output / "evaluation.json")
else:
    print("Synthetic smoke test: no model or report saved.")

## 10 · Summary, limitations and references

**What you can defend in a technical interview**
- You used a reproducible supervised-learning pipeline with separate holdout splits, a naive comparator and appropriate class metrics.
- You detected and documented contradictory annotations and prevented exact-text leakage across splits.
- You inspected errors and model features rather than treating a high headline classification score as evidence of financial predictive ability.

**What FinSight does *not* yet do:** ingest news feeds, retrieve annual reports, verify company-specific facts, predict price movements, perform valuation, or automate investment decisions. See [the roadmap](../docs/ROADMAP.md) for proposed extensions.

**Sources**
1. Malo, P. et al. (2014), *Good debt or bad debt: Detecting semantic orientations in economic texts*, JASIST. [Dataset collection](https://huggingface.co/datasets/lmassaron/FinancialPhraseBank).
2. Pedregosa, F. et al. (2011), *Scikit-learn: Machine Learning in Python*, JMLR. [Paper](https://jmlr.org/papers/v12/pedregosa11a.html).
3. Rule, A. et al. (2019), *Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks*, PLOS Computational Biology. [Paper](https://doi.org/10.1371/journal.pcbi.1007007).

**Dataset licence:** check the dataset provider's **CC BY-NC-SA 4.0** terms separately from this repository's MIT code licence. This notebook does not redistribute the dataset.